In [51]:
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC


# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv(
    "../data/kerala_crop_project_final_15000_recent.csv"
)


# ============================================================
# HANDLE MISSING VALUES
# ============================================================

for col in df.select_dtypes(include=["number"]).columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include=["object", "str"]).columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])


# ============================================================
# FEATURES
# ============================================================

features = [
    "Soil_Type",
    "Soil_pH",
    "Nitrogen_N_kg_ha",
    "Phosphorus_P_kg_ha",
    "Potassium_K_kg_ha",
    "Rainfall_mm",
    "Temperature_C",
    "Humidity_percent",
    "Irrigation_Level",
    "Season"
]

X = df[features].copy()
y = df["Crop"].astype(str)


# ============================================================
# ENCODE CATEGORICAL FEATURES
# ============================================================

for col in [
    "Soil_Type",
    "Irrigation_Level",
    "Season"
]:

    encoder = LabelEncoder()

    X[col] = encoder.fit_transform(
        X[col].astype(str)
    )


# ============================================================
# ENCODE TARGET
# ============================================================

target_encoder = LabelEncoder()

y = target_encoder.fit_transform(y)


# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)


# ============================================================
# SCALE FEATURES
# Helps Logistic Regression, KNN and SVM
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# ============================================================
# MODELS
# ============================================================

models = {


    "Decision Tree":
        (DecisionTreeClassifier(random_state=42),
         X_train,
         X_test),

    "Random Forest":
        (RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ),
         X_train,
         X_test),

    "KNN":
        (KNeighborsClassifier(n_neighbors=5),
         X_train_scaled,
         X_test_scaled),

    "SVM":
        (SVC(),
         X_train_scaled,
         X_test_scaled),


}


# ============================================================
# TRAIN AND SHOW ACCURACY ONLY
# ============================================================

print("=" * 50)
print("CROP RECOMMENDATION - ACCURACY")
print("=" * 50)

results = []

for name, (model, train_data, test_data) in models.items():

    model.fit(
        train_data,
        y_train
    )

    prediction = model.predict(
        test_data
    )

    accuracy = accuracy_score(
        y_test,
        prediction
    )

    results.append(
        [name, accuracy]
    )

    print(
        f"{name:<25} : {accuracy:.2%}"
    )


# ============================================================
# BEST MODEL
# ============================================================

results_df = pd.DataFrame(
    results,
    columns=["Algorithm", "Accuracy"]
)

best = results_df.loc[
    results_df["Accuracy"].idxmax()
]

print("\n" + "=" * 50)
print("BEST ALGORITHM")
print("=" * 50)

print("Algorithm :", best["Algorithm"])
print("Accuracy  :", f"{best['Accuracy']:.2%}")

CROP RECOMMENDATION - ACCURACY
Decision Tree             : 75.93%
Random Forest             : 83.62%
KNN                       : 79.18%
SVM                       : 83.98%

BEST ALGORITHM
Algorithm : SVM
Accuracy  : 83.98%


In [53]:
import pandas as pd
import warnings
import os
import joblib

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC


# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv(
    "../data/kerala_crop_project_final_15000_recent.csv"
)

print("Dataset loaded:", df.shape)


# ============================================================
# HANDLE MISSING VALUES
# ============================================================

for col in df.select_dtypes(include=["number"]).columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include=["object", "str"]).columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])


# ============================================================
# FEATURES
# ============================================================

features = [
    "Soil_Type",
    "Soil_pH",
    "Nitrogen_N_kg_ha",
    "Phosphorus_P_kg_ha",
    "Potassium_K_kg_ha",
    "Rainfall_mm",
    "Temperature_C",
    "Humidity_percent",
    "Irrigation_Level",
    "Season"
]

X = df[features].copy()
y = df["Crop"].astype(str)


# ============================================================
# ENCODE FEATURES
# ============================================================

feature_encoders = {}

for col in [
    "Soil_Type",
    "Irrigation_Level",
    "Season"
]:

    encoder = LabelEncoder()

    X[col] = encoder.fit_transform(
        X[col].astype(str)
    )

    feature_encoders[col] = encoder


# ============================================================
# ENCODE TARGET
# ============================================================

target_encoder = LabelEncoder()

y = target_encoder.fit_transform(y)


# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)


# ============================================================
# MODELS
# ============================================================

models = {

    "Decision Tree":
        DecisionTreeClassifier(
            random_state=42
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ),

    "KNN":
        KNeighborsClassifier(
            n_neighbors=5
        ),

    "SVM":
        SVC()
}


# ============================================================
# TRAIN AND SHOW ACCURACY
# ============================================================

print("\n")
print("=" * 60)
print("       CROP RECOMMENDATION - ACCURACY")
print("=" * 60)

results = {}

trained_models = {}


for name, model in models.items():

    model.fit(
        X_train,
        y_train
    )

    prediction = model.predict(
        X_test
    )

    accuracy = accuracy_score(
        y_test,
        prediction
    )

    results[name] = accuracy
    trained_models[name] = model

    print(
        f"{name:<25} : {accuracy:.2%}"
    )


# ============================================================
# RANDOM FOREST
# ============================================================

random_forest_model = trained_models[
    "Random Forest"
]

rf_accuracy = results[
    "Random Forest"
]


print("\n")
print("=" * 60)
print("             RANDOM FOREST")
print("=" * 60)

print(
    "Accuracy :",
    f"{rf_accuracy:.2%}"
)


# ============================================================
# SAVE EVERYTHING IN ONE FILE
# ============================================================

model_package = {

    "model": random_forest_model,

    "feature_encoders": feature_encoders,

    "target_encoder": target_encoder,

    "features": features
}


# ============================================================
# CREATE MODELS FOLDER
# ============================================================

save_folder = "../models"

os.makedirs(
    save_folder,
    exist_ok=True
)


# ============================================================
# SAVE MODEL
# ============================================================

model_path = os.path.join(
    save_folder,
    "crop_recommendation_model.pkl"
)

joblib.dump(
    model_package,
    model_path
)


# ============================================================
# FINAL
# ============================================================

print("\n")
print("=" * 60)
print("             MODEL SAVED")
print("=" * 60)

print(
    "Saved at:",
    model_path
)

Dataset loaded: (15000, 21)


       CROP RECOMMENDATION - ACCURACY
Decision Tree             : 75.93%
Random Forest             : 83.62%
KNN                       : 54.80%
SVM                       : 20.42%


             RANDOM FOREST
Accuracy : 83.62%


             MODEL SAVED
Saved at: ../models\crop_recommendation_model.pkl


In [50]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)


# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv(
    "../data/kerala_crop_project_final_15000_recent.csv"
)

print("Dataset loaded:", df.shape)


# ============================================================
# TARGET
# ============================================================

df["Yield_kg_ha"] = pd.to_numeric(
    df["Yield_kg_ha"],
    errors="coerce"
)

df = df.dropna(
    subset=["Yield_kg_ha"]
)


# ============================================================
# FEATURES
# ============================================================

features = [
    "Crop",
    "Soil_Type",
    "Soil_pH",
    "Nitrogen_N_kg_ha",
    "Phosphorus_P_kg_ha",
    "Potassium_K_kg_ha",
    "Rainfall_mm",
    "Temperature_C",
    "Humidity_percent",
    "Irrigation_Level",
    "Season"
]

X = df[features].copy()
y = df["Yield_kg_ha"].copy()


# ============================================================
# FEATURE TYPES
# ============================================================

categorical_features = [
    "Crop",
    "Soil_Type",
    "Irrigation_Level",
    "Season"
]

numeric_features = [
    "Soil_pH",
    "Nitrogen_N_kg_ha",
    "Phosphorus_P_kg_ha",
    "Potassium_K_kg_ha",
    "Rainfall_mm",
    "Temperature_C",
    "Humidity_percent"
]


# ============================================================
# MISSING VALUES
# ============================================================

for col in numeric_features:
    X[col] = X[col].fillna(
        X[col].median()
    )

for col in categorical_features:
    X[col] = X[col].fillna(
        X[col].mode()[0]
    )


# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)


# ============================================================
# ENCODING
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)


# ============================================================
# MODELS
# ============================================================

models = {

    "Ridge Regression":
        Ridge(
            alpha=1000
        ),

    "Decision Tree":
        DecisionTreeRegressor(
            max_depth=2,
            min_samples_split=350,
            min_samples_leaf=175,
            max_features=0.4,
            random_state=42
        ),

    "Random Forest":
        RandomForestRegressor(
            n_estimators=30,
            max_depth=3,
            min_samples_split=150,
            min_samples_leaf=75,
            max_features=0.3,
            random_state=42,
            n_jobs=-1
        ),

    "Gradient Boosting":
        GradientBoostingRegressor(
            n_estimators=50,
            max_depth=1,
            learning_rate=0.03,
            min_samples_split=200,
            min_samples_leaf=100,
            random_state=42
        )
}


# ============================================================
# TRAIN + EVALUATION
# ============================================================

print("\n")
print("=" * 75)
print("             CROP YIELD PREDICTION")
print("=" * 75)

results = []


for name, model in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    pipeline.fit(
        X_train,
        y_train
    )

    # --------------------------------------------------------
    # PREDICTION
    # --------------------------------------------------------

    prediction = pipeline.predict(
        X_test
    )

    # --------------------------------------------------------
    # METRICS
    # --------------------------------------------------------

    r2 = r2_score(
        y_test,
        prediction
    )

    mae = mean_absolute_error(
        y_test,
        prediction
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            prediction
        )
    )

    # --------------------------------------------------------
    # STORE RESULTS
    # --------------------------------------------------------

    results.append(
        [
            name,
            r2,
            mae,
            rmse
        ]
    )

    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    print(
        f"{name:<25} "
        f"R²: {r2:.4f}   "
        f"MAE: {mae:.2f}   "
        f"RMSE: {rmse:.2f}"
    )


# ============================================================
# RESULT DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results,
    columns=[
        "Algorithm",
        "R2 Score",
        "MAE",
        "RMSE"
    ]
)


# ============================================================
# DISPLAY RESULT
# ============================================================

print("\n")
print("=" * 75)
print("                         RESULT")
print("=" * 75)

print(
    results_df.to_string(
        index=False
    )
)


# ============================================================
# BEST ALGORITHM BASED ON R²
# ============================================================

best = results_df.loc[
    results_df["R2 Score"].idxmax()
]


print("\n")
print("=" * 75)
print("                    BEST ALGORITHM")
print("=" * 75)

print(
    "Algorithm :",
    best["Algorithm"]
)

print(
    "R² Score  :",
    f"{best['R2 Score']:.4f}"
)

print(
    "MAE       :",
    f"{best['MAE']:.2f}"
)

print(
    "RMSE      :",
    f"{best['RMSE']:.2f}"
)

Dataset loaded: (15000, 21)


             CROP YIELD PREDICTION
Ridge Regression          R²: 0.6409   MAE: 4027.85   RMSE: 6588.51
Decision Tree             R²: 0.6273   MAE: 3484.51   RMSE: 6711.37
Random Forest             R²: 0.8831   MAE: 2702.08   RMSE: 3758.44
Gradient Boosting         R²: 0.7141   MAE: 4391.20   RMSE: 5878.74


                         RESULT
        Algorithm  R2 Score         MAE        RMSE
 Ridge Regression  0.640866 4027.854582 6588.506286
    Decision Tree  0.627347 3484.506645 6711.367588
    Random Forest  0.883131 2702.076658 3758.441740
Gradient Boosting  0.714076 4391.198621 5878.739682


                    BEST ALGORITHM
Algorithm : Random Forest
R² Score  : 0.8831
MAE       : 2702.08
RMSE      : 3758.44


In [54]:
import pandas as pd
import numpy as np
import warnings
import os
import joblib

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)


# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv(
    "../data/kerala_crop_project_final_15000_recent.csv"
)

print("Dataset loaded:", df.shape)


# ============================================================
# TARGET
# ============================================================

df["Yield_kg_ha"] = pd.to_numeric(
    df["Yield_kg_ha"],
    errors="coerce"
)

df = df.dropna(
    subset=["Yield_kg_ha"]
)


# ============================================================
# FEATURES
# ============================================================

features = [
    "Crop",
    "Soil_Type",
    "Soil_pH",
    "Nitrogen_N_kg_ha",
    "Phosphorus_P_kg_ha",
    "Potassium_K_kg_ha",
    "Rainfall_mm",
    "Temperature_C",
    "Humidity_percent",
    "Irrigation_Level",
    "Season"
]

X = df[features].copy()

y = df["Yield_kg_ha"].copy()


# ============================================================
# FEATURE TYPES
# ============================================================

categorical_features = [
    "Crop",
    "Soil_Type",
    "Irrigation_Level",
    "Season"
]

numeric_features = [
    "Soil_pH",
    "Nitrogen_N_kg_ha",
    "Phosphorus_P_kg_ha",
    "Potassium_K_kg_ha",
    "Rainfall_mm",
    "Temperature_C",
    "Humidity_percent"
]


# ============================================================
# MISSING VALUES
# ============================================================

for col in numeric_features:
    X[col] = X[col].fillna(
        X[col].median()
    )

for col in categorical_features:
    X[col] = X[col].fillna(
        X[col].mode()[0]
    )


# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)


# ============================================================
# ENCODING
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)


# ============================================================
# MODELS
# ============================================================

models = {

    "Ridge Regression":
        Ridge(
            alpha=1000
        ),

    "Decision Tree":
        DecisionTreeRegressor(
            max_depth=2,
            min_samples_split=350,
            min_samples_leaf=175,
            max_features=0.4,
            random_state=42
        ),

    "Random Forest":
        RandomForestRegressor(
            n_estimators=30,
            max_depth=3,
            min_samples_split=150,
            min_samples_leaf=75,
            max_features=0.3,
            random_state=42,
            n_jobs=-1
        ),

    "Gradient Boosting":
        GradientBoostingRegressor(
            n_estimators=50,
            max_depth=1,
            learning_rate=0.03,
            min_samples_split=200,
            min_samples_leaf=100,
            random_state=42
        )
}


# ============================================================
# TRAIN + EVALUATION
# ============================================================

print("\n")
print("=" * 75)
print("             CROP YIELD PREDICTION")
print("=" * 75)

results = []

trained_pipelines = {}


for name, model in models.items():

    # --------------------------------------------------------
    # CREATE PIPELINE
    # --------------------------------------------------------

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "model",
                model
            )
        ]
    )

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    pipeline.fit(
        X_train,
        y_train
    )

    # --------------------------------------------------------
    # PREDICTION
    # --------------------------------------------------------

    prediction = pipeline.predict(
        X_test
    )

    # --------------------------------------------------------
    # METRICS
    # --------------------------------------------------------

    r2 = r2_score(
        y_test,
        prediction
    )

    mae = mean_absolute_error(
        y_test,
        prediction
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            prediction
        )
    )

    # --------------------------------------------------------
    # STORE
    # --------------------------------------------------------

    results.append(
        [
            name,
            r2,
            mae,
            rmse
        ]
    )

    trained_pipelines[name] = pipeline

    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    print(
        f"{name:<25} "
        f"R²: {r2:.4f}   "
        f"MAE: {mae:.2f}   "
        f"RMSE: {rmse:.2f}"
    )


# ============================================================
# RESULT DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results,
    columns=[
        "Algorithm",
        "R2 Score",
        "MAE",
        "RMSE"
    ]
)


# ============================================================
# DISPLAY RESULT
# ============================================================

print("\n")
print("=" * 75)
print("                         RESULT")
print("=" * 75)

print(
    results_df.to_string(
        index=False
    )
)


# ============================================================
# BEST ALGORITHM
# ============================================================

best = results_df.loc[
    results_df["R2 Score"].idxmax()
]

best_algorithm = best["Algorithm"]

best_pipeline = trained_pipelines[
    best_algorithm
]


# ============================================================
# BEST MODEL DETAILS
# ============================================================

print("\n")
print("=" * 75)
print("                    BEST ALGORITHM")
print("=" * 75)

print(
    "Algorithm :",
    best_algorithm
)

print(
    "R² Score  :",
    f"{best['R2 Score']:.4f}"
)

print(
    "MAE       :",
    f"{best['MAE']:.2f}"
)

print(
    "RMSE      :",
    f"{best['RMSE']:.2f}"
)


# ============================================================
# CREATE MODELS FOLDER
# ============================================================

save_folder = "../models"

os.makedirs(
    save_folder,
    exist_ok=True
)


# ============================================================
# SAVE BEST MODEL + PREPROCESSOR
# ============================================================

model_path = os.path.join(
    save_folder,
    "crop_yield_prediction_model.pkl"
)

joblib.dump(
    best_pipeline,
    model_path
)


# ============================================================
# FINAL MESSAGE
# ============================================================

print("\n")
print("=" * 75)
print("                  MODEL SAVED")
print("=" * 75)

print(
    "Saved model:",
    model_path
)

print(
    "\nThe saved file contains:"
)

print(
    "1. Preprocessing / OneHotEncoder"
)

print(
    "2. Trained",
    best_algorithm
)

print(
    "3. All required transformations"
)

print("\nModel saved successfully!")

Dataset loaded: (15000, 21)


             CROP YIELD PREDICTION
Ridge Regression          R²: 0.6409   MAE: 4027.85   RMSE: 6588.51
Decision Tree             R²: 0.6273   MAE: 3484.51   RMSE: 6711.37
Random Forest             R²: 0.8831   MAE: 2702.08   RMSE: 3758.44
Gradient Boosting         R²: 0.7141   MAE: 4391.20   RMSE: 5878.74


                         RESULT
        Algorithm  R2 Score         MAE        RMSE
 Ridge Regression  0.640866 4027.854582 6588.506286
    Decision Tree  0.627347 3484.506645 6711.367588
    Random Forest  0.883131 2702.076658 3758.441740
Gradient Boosting  0.714076 4391.198621 5878.739682


                    BEST ALGORITHM
Algorithm : Random Forest
R² Score  : 0.8831
MAE       : 2702.08
RMSE      : 3758.44


                  MODEL SAVED
Saved model: ../models\crop_yield_prediction_model.pkl

The saved file contains:
1. Preprocessing / OneHotEncoder
2. Trained Random Forest
3. All required transformations

Model saved successfully!
